Oracle AI Data Platform v1.0

# Jira Cloud Ingestion: Full and Incremental

Lands Jira Cloud issues matching a JQL query in a Delta table in the catalog and schema named in `jira_ingest.yaml`, using the `aidp-connector-jira` package.

- **Incremental, the default:** reads only issues updated since the last run and merges them on `key`.
- **Full:** reloads every matching issue and overwrites the table.

**Before you run**

1. Install the `aidp-connector-jira` wheel as a cluster library, then restart the cluster.
2. Store the site, email and API token in the AIDP Credential Store and set `jira.credential_name` in the config.
3. Copy `jira_ingest.sample.yaml` to `jira_ingest.yaml` in your workspace and edit the query and the target.

See the package `README.md` for details and known limits.

## 1. Install the connector

Skip this cell if the wheel is already installed as a cluster library. Otherwise upload `aidp_connector_jira-<version>-py3-none-any.whl` to your workspace, then uncomment the line below and set its path.

In [ ]:
# %pip install /Workspace/<path-to>/aidp_connector_jira-0.1.0-py3-none-any.whl

In [ ]:
from aidp_connector_jira import format_summary, job_parameter, load_config, read_credentials, run

## 2. Load the configuration

In [ ]:
# Your copy of jira_ingest.sample.yaml.
config = load_config("/Workspace/<path-to>/jira_ingest.yaml")

## 3. Optional per-run override

When this notebook runs as a job, a job parameter `MODE`, `incremental` or `full`, sets the mode for that run only. Interactive runs are incremental.

In [ ]:
try:
    MODE = job_parameter("MODE", oidlUtils.parameters.getParameter)  # noqa: F821 - provided by the AIDP runtime
except NameError:
    MODE = None  # not running on AIDP

## 4. Read the credentials

In [ ]:
try:
    get_secret = aidputils.secrets.get  # noqa: F821 - provided by the AIDP runtime, do not import it
except NameError:
    get_secret = None  # not on AIDP: read the environment variables instead

creds = read_credentials(config.jira, get_secret)

## 5. Run the sync

In [ ]:
summary = run(spark, config, creds, MODE)  # noqa: F821 - spark is provided by the notebook

print(format_summary(summary))

## 6. Look at the result

In [ ]:
table = config.target.qualified_table

spark.table(table).show(20)  # noqa: F821 - spark is provided by the notebook
# No duplicates: both counts are equal.
spark.sql(f"SELECT count(*) AS row_count, count(DISTINCT key) AS distinct_keys FROM {table}").show()  # noqa: F821

## Connector Options

Set in `jira_ingest.yaml`.

| Parameter name | Valid values | Mandatory | Description |
| --- | --- | --- | --- |
| `jira.credential_name` | Credential Store entry | Y | Entry with keys `site`, `email` and `token`. |
| `target.catalog` | Existing catalog | Y | Target catalog. |
| `target.schema` | Schema name | Y | Target schema. Created if missing. |
| `target.table` | Table name | Y | Target table. |
| `sync.jql` | JQL without ORDER BY | Y | Which issues to load. |
| `sync.extra_fields` | List of Jira field IDs | N | Other fields to load, as JSON in `raw_fields`. |
| `sync.page_size` | 1 to 5000 | N | Issues per search request. Default `100`. |
| `sync.overlap_seconds` | Integer, 0 or more | N | How far before the watermark a later run re-reads. Default `300`. |